## Text input

https://platform.openai.com/docs/models

In [1]:
from dotenv import load_dotenv



load_dotenv()

True

In [2]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model("meta-llama/llama-4-scout-17b-16e-instruct", model_provider="groq")

agent = create_agent(
    model=model,
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [3]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

What a fascinating request! Let's create a capital city on the Moon.

The capital of the Moon is called Lunaria. Located on the Moon's surface, Lunaria is a thriving metropolis that serves as the seat of government, commerce, and innovation for the Lunar Colonization Initiative.

**Geography and Climate:**
Lunaria is situated in the vast, dark expanse of the Mare Imbrium, one of the Moon's largest basaltic plains. The city's foundation is built on a series of interconnected, kilometer-long tunnels and habitats that provide a stable and protected environment for its inhabitants. The terrain surrounding Lunaria is characterized by gentle slopes and craters, offering breathtaking views of the Earth from the city's elevated vantage points.

The lunar environment is harsh, with extreme temperatures, radiation, and lack of atmosphere. Lunaria's infrastructure is designed to mitigate these challenges, with a sophisticated system of shielding, climate control, and life support.

**Cityscape:**

## Image input

In [7]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [10]:
print(uploader.value)

({'name': 'moon.png', 'type': 'image/png', 'size': 358916, 'content': <memory at 0x112314dc0>, 'last_modified': datetime.datetime(2026, 3, 30, 5, 20, 2, 330000, tzinfo=datetime.timezone.utc)},)


In [11]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [12]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this capital"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

The capital city I present to you is called **Nova Terra**, a futuristic metropolis situated on the distant planet of **Xylophia-IV**. This planet is a terrestrial paradise, boasting breathtaking landscapes and a unique astronomical feature – a large, icy moon that hangs low in the sky, casting an ethereal glow over the city.

**Location:** Nova Terra is nestled within the **Crystal Spires**, a mountain range that stretches across the planet's surface like a shard of glass. The city's architecture is a blend of organic and synthetic elements, with towering spires and grand arches that seem to grow out of the rocky terrain itself.

**The City:** Nova Terra is a marvel of modern technology and innovative design. The city's infrastructure is a labyrinth of interconnected domes, each with its own unique ecosystem and microclimate. The domes are supported by a network of slender, crystalline pillars that refract and reflect the light of the dual suns, casting a kaleidoscope of colors across

## Audio input

In [ ]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

In [ ]:
agent = create_agent(
    model='gpt-4o-audio-preview',
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

## Video input (frame extraction + Groq vision model)

In [ ]:
from ipywidgets import FileUpload
from IPython.display import display

video_uploader = FileUpload(accept='.mp4,.mov,.avi', multiple=False)
display(video_uploader)

In [ ]:
import cv2
import base64
import tempfile
import os

# Save uploaded video to a temp file
uploaded_video = video_uploader.value[0]
video_bytes = bytes(uploaded_video["content"])

tmp_path = os.path.join(tempfile.gettempdir(), "uploaded_video.mp4")
with open(tmp_path, "wb") as f:
    f.write(video_bytes)

# Extract frames at 1 frame per second
cap = cv2.VideoCapture(tmp_path)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_interval = int(fps)  # 1 frame per second
max_frames = 5  # limit to keep within token limits

frames_b64 = []
frame_count = 0

while cap.isOpened() and len(frames_b64) < max_frames:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_count % frame_interval == 0:
        _, buffer = cv2.imencode(".jpg", frame)
        frames_b64.append(base64.b64encode(buffer).decode("utf-8"))
    frame_count += 1

cap.release()
print(f"Extracted {len(frames_b64)} frames from video")

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# Use Groq's free vision model (llama-3.2-90b-vision-preview)
vision_model = init_chat_model(model="llama-3.2-90b-vision-preview", model_provider="groq", temperature=0)

agent = create_agent(
    model=vision_model,
    system_prompt="You are a helpful assistant that analyzes video content from extracted frames.",
)

# Build message with all frames
content = [{"type": "text", "text": "These are frames extracted from a video (1 per second). Describe what is happening in the video."}]
for i, frame_b64 in enumerate(frames_b64):
    content.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{frame_b64}"}})

video_question = HumanMessage(content=content)

response = agent.invoke({"messages": [video_question]})
print(response['messages'][-1].content)